# Data Preprocessing: 
## Merging and Selecting Features

## Objective
Create a unified dataset from **three data sources** with only the most relevant features for analysis and modeling.

## Data Sources
1. **Automobile.tn New Cars** - New vehicle listings
2. **Automobile.tn Used Cars** - Pre-owned vehicle listings  
3. **Tayara.tn** - Marketplace used car listings

## Steps

1. **Merge Datasets**
   - Combine all three datasets.
   - Add a new column `new`:
     - `"yes"` for new cars (automobile.tn)  
     - `"no"` for used cars (automobile.tn & tayara.tn)  

2. **Select Relevant Features**
   - Columns retained based on previous **Data Quality Assessment** and feature availability across all sources:
     - **Price** – convert new cars to numeric  
     - **Brand**  
     - **Model**  
     - **Kilométrage** – set to `0` for new cars  
     - **Body type** (`Carrosserie` / `Type de carrosserie`)  
     - **Fuel type** (`Energie` / `Carburant`)  
     - **Transmission** (`Boîte` / `Boite vitesse` / `Boite`)  
     - **Puissance fiscale** – used instead of `Puissance en chevaux` due to missing data  
     - **Date mise en circulation** / **Année** – set to `2025` for new cars  
     - **New flag** – added in merging step  

     - Redundancy (e.g., number of doors vs. body type, car dimensions vs. body type)  
   - Columns removed due to:
     - **Not available across all data sources** (seats, climatisation not in Tayara data)

## Outcome
A clean, consolidated dataset combining automobile.tn (new & used) and Tayara.tn marketplace data with consistent features across all sources, ready for preprocessing steps like encoding, missing value handling, and scaling.


## Outcome
A clean, consolidated dataset with key features, ready for preprocessing steps like encoding, missing value handling, and scaling.

In [112]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load data from all three sources
df_new = pd.read_csv("../data_scraper/data_scraped/automobileTnNew.csv")
df_used = pd.read_csv("../data_scraper/data_scraped/automobileTnUsed.csv")
df_tayara = pd.read_csv("../data_scraper/data_scraped/tayara_cars.csv")


def clean_year_string(year_val):
    """Ensure automobile.tn year strings stay in month.year format."""
    if pd.isna(year_val):
        return None
    year_str = str(year_val)
    if '.' in year_str:
        parts = year_str.split('.')
        if len(parts) == 2:
            month = parts[0]
            year_part = parts[1]
            if len(year_part) == 3:
                year_part = year_part + '0'
            return f"{month}.{year_part}"
    return year_str


def tayara_year_to_registration(year_val):
    """Convert Tayara year (Année) to month.year string (month=6 default)."""
    if pd.isna(year_val):
        return None
    try:
        year = int(float(year_val))
        if 1990 <= year <= 2025:
            return f"6.{year}"
    except Exception:
        pass
    return None


df_used['Mise en circulation'] = df_used['Mise en circulation'].apply(clean_year_string)

df_used['new'] = 'no'
df_tayara['new'] = 'no'
df_new['new'] = 'yes'

# -------------------- New cars --------------------
df_new['Price_clean'] = df_new['Price'].str.replace(' ', '').astype(float)
df_new['Kilométrage'] = 0
df_new['Mise en circulation'] = '12.2025'

df_new_clean = df_new[[
    'Brand', 'Model', 'Price_clean', 'Kilométrage', 'Carrosserie',
    'Energie', 'Boîte', 'Puissance fiscale', 'Mise en circulation', 'new'
]].copy()

df_new_clean = df_new_clean.rename(columns={
    'Brand': 'brand',
    'Model': 'model',
    'Price_clean': 'price',
    'Kilométrage': 'kilometrage',
    'Carrosserie': 'body_type',
    'Energie': 'fuel_type',
    'Boîte': 'transmission',
    'Puissance fiscale': 'puissance_fiscale',
    'Mise en circulation': 'registration_year'
})

# -------------------- Used cars --------------------
df_used_clean = df_used[[
    'Spécifications - Marque', 'Spécifications - Modèle', 'Price',
    'Kilométrage', 'Carrosserie', 'Motorisation - Énergie',
    'Boite vitesse', 'Puissance fiscale', 'Mise en circulation', 'new'
]].copy()

df_used_clean = df_used_clean.rename(columns={
    'Spécifications - Marque': 'brand',
    'Spécifications - Modèle': 'model',
    'Price': 'price',
    'Kilométrage': 'kilometrage',
    'Carrosserie': 'body_type',
    'Motorisation - Énergie': 'fuel_type',
    'Boite vitesse': 'transmission',
    'Puissance fiscale': 'puissance_fiscale',
    'Mise en circulation': 'registration_year'
})

# -------------------- Tayara cars --------------------
df_tayara['registration_year'] = df_tayara['Année'].apply(tayara_year_to_registration)

df_tayara_clean = df_tayara[[
    'Marque', 'Modèle', 'Price', 'Kilométrage', 'Type de carrosserie',
    'Carburant', 'Boite', 'Puissance fiscale', 'registration_year', 'new'
]].copy()

df_tayara_clean = df_tayara_clean.rename(columns={
    'Marque': 'brand',
    'Modèle': 'model',
    'Price': 'price',
    'Kilométrage': 'kilometrage',
    'Type de carrosserie': 'body_type',
    'Carburant': 'fuel_type',
    'Boite': 'transmission',
    'Puissance fiscale': 'puissance_fiscale'
})

# -------------------- Merge all sources --------------------
merged_df = pd.concat([df_used_clean, df_new_clean, df_tayara_clean], ignore_index=True)

print(f"Used cars (automobile.tn): {len(df_used_clean)}")
print(f"New cars (automobile.tn): {len(df_new_clean)}")
print(f"Tayara listings: {len(df_tayara_clean)}")
print(f"Final merged dataset shape: {merged_df.shape}")

print("\nSample rows:")
print(merged_df.head())

Used cars (automobile.tn): 2148
New cars (automobile.tn): 510
Tayara listings: 4029
Final merged dataset shape: (6687, 10)

Sample rows:
           brand         model     price kilometrage body_type  \
0            GWM  Haval Jolion   78000.0  113 000 km       SUV   
1           Audi            A6   75000.0  160 000 km   Berline   
2  Mercedes-Benz     GLE Coupé  460000.0   15 000 km       SUV   
3  Mercedes-Benz      Classe E  118000.0  140 000 km   Berline   
4      Chevrolet        Groove   68000.0   67 000 km       SUV   

                      fuel_type transmission puissance_fiscale  \
0                       Essence  Automatique              9 cv   
1                       Essence  Automatique             10 cv   
2  Hybride rechargeable essence  Automatique             15 cv   
3                        Diesel  Automatique             10 cv   
4                       Essence  Automatique              6 cv   

  registration_year new  
0            7.2021  no  
1            1.20

## Cleaning Rows and Removing Outliers

## Objective
Refine the dataset by removing irrelevant or extreme entries to improve quality and consistency.

## Steps

1. **Filter by Fuel Type**
   - Keep only cars with fuel type:
     - `Essence` (petrol)
     - `Diesel`
     - `Hybride` (hybrid)
     - `Electrique` (electric)
   - combine all variations of hybrid into one hybrid class.

2. **Filter by Brand and Body Type**
   - Remove cars and brands with **very low representation** to avoid sparsity and unreliable patterns.
   - Apply the same principle to **body types** to focus on the most common categories.

3. **Remove Outliers**
   - **Price**: remove extreme low or high values outside expected range.  
   - **Year of circulation** (`Date mise en circulation`): remove unrealistic years.  
   - **Kilométrage**: remove extreme mileage values that are likely data errors.

## Outcome
A cleaner, more representative dataset with consistent fuel types, brands, body types, and realistic numerical values, ready for further preprocessing like encoding and scaling.


In [113]:
# DATA CLEANING And ROW FILTERING

print(f"Initial dataset shape: {merged_df.shape}")

# Create a clean copy
merged_clean = merged_df.copy()

# 1. Fuel Type Transformation : simplify categories and remove Compacte (it has one row only)
fuel_counts = merged_clean['fuel_type'].value_counts()
print("\nFuel type distribution:")
print(fuel_counts)

# Remove the "Compacte" row first
merged_clean = merged_clean[merged_clean['fuel_type'] != 'Compacte'].copy()

# Transform fuel types into simplified categories
def simplify_fuel_type(fuel):
    fuel = str(fuel).lower()
    if 'electrique' in fuel:
        return 'electrique'
    elif 'hybride' in fuel:
        return 'hybride' 
    else:
        return fuel  

merged_clean['fuel_type_simple'] = merged_clean['fuel_type'].apply(simplify_fuel_type)

# Keep all simplified fuel types except rare ones
fuel_types_to_keep = ['essence', 'diesel', 'hybride', 'electrique']
merged_clean = merged_clean[merged_clean['fuel_type_simple'].isin(fuel_types_to_keep)].copy()
print(f"After fuel filtering: {merged_clean.shape}")

print("\nSimplified fuel type distribution:")
print(merged_clean['fuel_type_simple'].value_counts())

# 2. Brand Filtering : remove brands with low representation
brand_counts = merged_clean['brand'].value_counts()
print(f"\nBrand counts:")
print(brand_counts)

# Keep brands with at least 10 cars
brands_to_keep = brand_counts[brand_counts >= 10].index
merged_clean = merged_clean[merged_clean['brand'].isin(brands_to_keep)].copy()
print(f"After brand filtering: {merged_clean.shape}")

# 3. Body Type Filtering : remove rare body types
body_counts = merged_clean['body_type'].value_counts()
print(f"\nBody type counts:")
print(body_counts)

# Keep body types with at least 30 cars
bodies_to_keep = body_counts[body_counts >= 30].index
merged_clean = merged_clean[merged_clean['body_type'].isin(bodies_to_keep)].copy()
print(f"After body type filtering: {merged_clean.shape}")

# 4. Price Outlier Removal
price_stats = merged_clean['price'].describe()
print(f"\nPrice statistics before:")
print(price_stats)

# Remove extreme price outliers (bottom and top 1%)
Q1_price = merged_clean['price'].quantile(0.01)
Q3_price = merged_clean['price'].quantile(0.99)
merged_clean = merged_clean[(merged_clean['price'] >= Q1_price) & (merged_clean['price'] <= Q3_price)].copy()
print(f"After price filtering: {merged_clean.shape}")


Initial dataset shape: (6687, 10)

Fuel type distribution:
Essence                           4423
Diesel                            1721
Hybride rechargeable essence       127
Electrique                         106
Hybride essence                     77
Hybride léger essence               60
Essence | Hybride rechargeable      34
Essence | Hybride léger             25
Essence | Hybride                   23
Hybride Diesel                      16
Hybride léger diesel                10
Hybride                              8
Hybride rechargeable diesel          3
Diesel | Hybride léger               2
Compacte                             1
Name: fuel_type, dtype: int64
After fuel filtering: (6635, 11)

Simplified fuel type distribution:
essence       4423
diesel        1721
hybride        385
electrique     106
Name: fuel_type_simple, dtype: int64

Brand counts:
Volkswagen       722
Mercedes-Benz    649
Peugeot          560
Renault          380
Citroen          308
                ... 
Lad

In [114]:
# 5. Year Filtering : 
print(f"=== YEAR FILTERING ===")

# Extract year from registration_year
def extract_year_fixed(year_str):
    if pd.isna(year_str):
        return None
    year_str = str(year_str).strip()
    if '.' in year_str:
        parts = year_str.split('.')
        if len(parts) == 2:
            year_part = parts[1]
            try:
                year = int(year_part)
                if 1900 <= year <= 2030:
                    return year
            except:
                return None
    return None

merged_clean['year_extracted'] = merged_clean['registration_year'].apply(extract_year_fixed)

# Remove cars outside 2006-2025 range and null years
current_year = 2025
initial_count = len(merged_clean)
merged_clean = merged_clean[
    (merged_clean['year_extracted'] >= 2006) & 
    (merged_clean['year_extracted'] <= current_year) &
    (merged_clean['year_extracted'].notnull())
].copy()

print(f"Rows removed by year filtering: {initial_count - len(merged_clean)}")
print(f"Final dataset shape: {merged_clean.shape}")

=== YEAR FILTERING ===
Rows removed by year filtering: 470
Final dataset shape: (5440, 12)


In [115]:
# 6. Mileage Outlier Removal 

# Clean kilometrage: remove non-numeric chars except minus and dot, then convert to numeric
kil_clean = merged_clean['kilometrage'].astype(str).str.replace(r'[^\d\.-]', '', regex=True)
kil_clean = kil_clean.replace('', np.nan)  # empty -> NaN
merged_clean['kilometrage'] = pd.to_numeric(kil_clean, errors='coerce')

before_count = len(merged_clean)
cond_keep = (
    ((merged_clean['kilometrage'] >= 0) & (merged_clean['kilometrage'] <= 300000)) |
    (merged_clean['new'] == 'yes')
)
merged_clean = merged_clean[cond_keep].copy()
print(f"After mileage filtering: {merged_clean.shape} (kept {len(merged_clean)} of {before_count})")

# summary
print(f"\n=== CLEANING SUMMARY ===")
print(f"Initial rows: {len(merged_df)}")
print(f"Final rows: {len(merged_clean)}")
print(f"Rows removed: {len(merged_df) - len(merged_clean)}")
print(f"Removal percentage: {((len(merged_df) - len(merged_clean)) / len(merged_df) * 100):.1f}%")

merged_clean.to_csv('car_prices_cleaned.csv', index=False)


After mileage filtering: (5193, 12) (kept 5193 of 5440)

=== CLEANING SUMMARY ===
Initial rows: 6687
Final rows: 5193
Rows removed: 1494
Removal percentage: 22.3%


In [116]:
#checking for empty values
print(merged_clean.isnull().sum())

# Checking categorical variables
print("Categorical variables value counts:")
for col in ['brand', 'body_type', 'fuel_type_simple', 'transmission', 'new']:
    print(f"\n{col}:")
    print(merged_clean[col].value_counts())


brand                  0
model                  0
price                  0
kilometrage            0
body_type              0
fuel_type              0
transmission          23
puissance_fiscale    110
registration_year      0
new                    0
fuel_type_simple       0
year_extracted         0
dtype: int64
Categorical variables value counts:

brand:
Volkswagen       558
Mercedes-Benz    556
Peugeot          425
BMW              248
Audi             238
Renault          235
Citroen          229
Kia              221
Hyundai          212
Ford             186
KIA              161
Fiat             148
Toyota           123
Seat             118
Suzuki            96
Chery             95
MG                93
Land Rover        89
Nissan            87
Dacia             67
Ssangyong         56
Mazda             54
Citroën           52
GWM               49
Opel              47
Porsche           46
Jeep              45
Mahindra          43
bmw               39
Skoda             38
Chevrolet    

## Standardization and Feature Engineering

##  Brand Name Standardization
- Correct inconsistencies in brand names using a mapping dictionary.
- Example: `'bmw' → 'BMW'`, `'mercedes-benz' → 'Mercedes-Benz'`.

##  Fuel Type Update
- Remove the old `fuel_type` column.
- Rename `fuel_type_simple` to `fuel_type` for clarity.

##  Puissance Fiscale Conversion
- Convert `puissance_fiscale` from string to numeric.
- Extract the first number from the string for accurate calculations.

##  Feature Engineering
- `car_age`: `2025 - year_extracted` to calculate vehicle age.
- `is_new`: binary flag for new cars (`1` = yes, `0` = no).
- `price_per_fiscal`: computed as `price / puissance_fiscale` for normalized pricing.

##  Binary Categoricals Conversion
- Convert categorical columns to 0/1:
  - `new`: `yes = 1`, `no = 0`
  - `transmission`: `Automatique = 1`, `Manuelle = 0`

In [117]:
# fix brand name inconsistencies
brand_mapping = {
    'bmw': 'BMW', 'mercedes-benz': 'Mercedes-Benz', 'hyundai': 'Hyundai',
    'kia': 'KIA', 'peugeot': 'Peugeot', 'toyota': 'Toyota',
    'volkswagen': 'Volkswagen', 'skoda': 'Skoda', 'suzuki': 'Suzuki',
    'renault': 'Renault', 'audi': 'Audi', 'volvo': 'Volvo',
    'seat': 'Seat', 'honda': 'Honda', 'fiat': 'Fiat', 'citroen': 'Citroën',
    'porsche': 'Porsche', 'mg': 'MG'
}

merged_clean['brand'] = merged_clean['brand'].replace(brand_mapping)

#  Remove old fuel_type and rename fuel_type_simple
merged_clean = merged_clean.drop('fuel_type', axis=1)
merged_clean = merged_clean.rename(columns={'fuel_type_simple': 'fuel_type'})

def extract_fiscal_power(value):
    if pd.isna(value):
        return None
    value_str = str(value)
    # Extract first number from string (ex : 5 CV -> 5)
    import re
    numbers = re.findall(r'\d+', value_str)
    if numbers:
        return float(numbers[0])
    return None

# 5. Convert puissance_fiscale to numeric
merged_clean['puissance_fiscale'] = merged_clean['puissance_fiscale'].apply(extract_fiscal_power)

# 6. Feature Engineering
merged_clean['car_age'] = 2025 - merged_clean['year_extracted']
merged_clean['is_new'] = (merged_clean['new'] == 'yes').astype(int)
merged_clean['price_per_fiscal'] = merged_clean['price'] / merged_clean['puissance_fiscale']

if 'month' in merged_clean.columns:
    merged_clean = merged_clean.drop(columns=['month'])

print(f"Dataset shape: {merged_clean.shape}")
print(f"New columns: car_age, is_new, price_per_fiscal")
print(f"Columns: {merged_clean.sample(10)}")

Dataset shape: (5193, 14)
New columns: car_age, is_new, price_per_fiscal
Columns:               brand        model     price  kilometrage   body_type  \
3224  Mercedes-Benz   Classe CLA  115000.0      52000.0     Berline   
519   Mercedes-Benz     Classe C  147000.0      90000.0     Berline   
5343           Fiat        Doblo   18000.0          0.0  Utilitaire   
2837        Citroen           C3   22000.0     191222.0     Berline   
2803            Kia     Sportage   94000.0     134000.0     Berline   
2111     Volkswagen       Tiguan   59800.0     128000.0         SUV   
4625       Mahindra        Hover   28000.0      92000.0    Compacte   
5013        Citroen           C3   35750.0      13000.0     Berline   
4427  Mercedes-Benz     Classe E   58000.0     270000.0    Compacte   
598             KIA  Rio Berline   49900.0      90000.0     Berline   

     transmission  puissance_fiscale registration_year new fuel_type  \
3224  Automatique                8.0            6.2021  no   ess

In [118]:
# Convert binary categoricals to 0 or 1
binary_mapping = {
    'new': {'yes': 1, 'no': 0},
    'transmission': {'Automatique': 1, 'Manuelle': 0}
}

for col, mapping in binary_mapping.items():
    merged_clean[col] = merged_clean[col].map(mapping)

# checking conversions
print("Binary conversions:")
for col in ['new', 'transmission']:
    print(f"{col}: {merged_clean[col].value_counts()}")

print(f"\nDataset shape: {merged_clean.shape}")
print(f"Columns: {merged_clean.sample(10)}")

Binary conversions:
new: 0    4862
1     331
Name: new, dtype: int64
transmission: 0.0    2699
1.0    2471
Name: transmission, dtype: int64

Dataset shape: (5193, 14)
Columns:            brand                model     price  kilometrage   body_type  \
2877  Alfa Romeo            Giulietta   44000.0      94000.0    Compacte   
1907  Land Rover   Range Rover Evoque   96000.0     176000.0         SUV   
1423         BMW              Série 3   78000.0     135000.0     Berline   
3332       Haval                   H6   62000.0     103000.0       4 x 4   
3755     Citroen             Berlingo   45000.0     210000.0  Utilitaire   
1024      Toyota  Yaris Cross Hybride  109000.0      10000.0         SUV   
3422  Volkswagen               Passat   32500.0        190.0     Berline   
3700   Chevrolet                 Aveo   23000.0     141000.0     Berline   
5136  Volkswagen               Tiguan   98000.0     138000.0      Autres   
3500     Peugeot                  208   25900.0     108000.0    

## Encoding, Scaling, and Final Cleaning

##  Target Encoding
- Encode categorical variables (`brand`, `body_type`, `model`) using mean `price` per category.
- Created separate CSVs for reference:
  - `brand_encoding.csv`
  - `body_type_encoding.csv`
  - `model_encoding.csv`
- Original columns dropped after encoding to avoid redundancy.

##  Feature Scaling
- Standardized numerical features for modeling using `StandardScaler`:
  - `kilometrage`, `puissance_fiscale`, `car_age`, `price_per_fiscal`
  - Encoded features: `brand_encoded`, `body_type_encoded`, `model_encoded`
- Ensures features are comparable in scale.

##  One-Hot Encoding
- Applied one-hot encoding to `fuel_type` for categorical modeling.
- Prefix `fuel_` added to new columns.

##  Redundant Column Removal
- Dropped columns replaced by engineered features:
  - `year_extracted` → `car_age`
  - `new` → `is_new`
  - `registration_year` removed after deriving year-based features
- Only columns existing in the dataset were dropped.

##  Missing Value Handling
- Identified rows with missing values.
- Only 6 rows out of 2230 had missing data.
- Dropped these rows to maintain clean dataset.

##  Final Dataset
- Preprocessed, scaled, and encoded dataset ready for modeling.
- Saved to `car_prices_final_preprocessed.csv`. 

In [119]:
def manual_target_encode(series, target):
    return series.map(target.groupby(series).mean())

brand_encoding_map = merged_clean.groupby('brand')['price'].mean().to_dict()
body_type_encoding_map = merged_clean.groupby('body_type')['price'].mean().to_dict()
model_encoding_map = merged_clean.groupby('model')['price'].mean().to_dict()

brand_encoding_df = pd.DataFrame({
    'brand': list(brand_encoding_map.keys()),
    'brand_encoded': list(brand_encoding_map.values())
})

body_type_encoding_df = pd.DataFrame({
    'body_type': list(body_type_encoding_map.keys()),
    'body_type_encoded': list(body_type_encoding_map.values())
})

model_encoding_df = pd.DataFrame({
    'model': list(model_encoding_map.keys()),
    'model_encoded': list(model_encoding_map.values())
})

brand_encoding_df.to_csv('brand_encoding.csv', index=False)
body_type_encoding_df.to_csv('body_type_encoding.csv', index=False)
model_encoding_df.to_csv('model_encoding.csv', index=False)

merged_clean['brand_encoded'] = manual_target_encode(merged_clean['brand'], merged_clean['price'])
merged_clean['body_type_encoded'] = manual_target_encode(merged_clean['body_type'], merged_clean['price'])
merged_clean['model_encoded'] = manual_target_encode(merged_clean['model'], merged_clean['price'])

merged_clean = merged_clean.drop(['brand', 'body_type', 'model', 'registration_year'], axis=1)

print(f"Dataset shape: {merged_clean.shape}")

#remove null values lines in puissance_fiscale
merged_clean = merged_clean[merged_clean['puissance_fiscale'].notnull()].copy()
#print head 10 then show cols and theyre cnull counts
print(merged_clean.head(10))
print(merged_clean.isnull().sum())

Dataset shape: (5193, 13)
       price  kilometrage  transmission  puissance_fiscale  new fuel_type  \
0    78000.0     113000.0           1.0                9.0    0   essence   
1    75000.0     160000.0           1.0               10.0    0   essence   
3   118000.0     140000.0           1.0               10.0    0    diesel   
4    68000.0      67000.0           1.0                6.0    0   essence   
5    47000.0     115000.0           0.0                4.0    0   essence   
6    44000.0     180000.0           0.0                7.0    0   essence   
7    42000.0      98000.0           0.0                5.0    0   essence   
8    44000.0     120000.0           0.0                8.0    0    diesel   
9    46000.0     146000.0           0.0                5.0    0   essence   
10   69000.0     132000.0           1.0                6.0    0   essence   

    year_extracted  car_age  is_new  price_per_fiscal  brand_encoded  \
0           2021.0      4.0       0       8666.666667 

In [120]:
from sklearn.preprocessing import StandardScaler

cols_to_scale = [
    'kilometrage',
    'puissance_fiscale',
    'car_age',
    'price_per_fiscal',
    'brand_encoded',
    'body_type_encoded',
    'model_encoded'
]

scaler = StandardScaler()

# Replace inf values, then fill NaNs with column medians before scaling
scale_block = merged_clean[cols_to_scale].replace([np.inf, -np.inf], np.nan)
scale_block = scale_block.fillna(scale_block.median())

merged_clean[cols_to_scale] = scaler.fit_transform(scale_block)

print("Scaled dataset sample:")
print(merged_clean.sample(5).round(3))

Scaled dataset sample:
         price  kilometrage  transmission  puissance_fiscale  new fuel_type  \
1659   71000.0       -0.588           0.0             -0.015    0   essence   
100   179000.0       -1.125           1.0             -0.014    0   hybride   
58     48000.0        0.048           0.0             -0.015    0   essence   
971   139000.0       -0.550           1.0             -0.014    0   essence   
1707   52000.0        0.290           1.0             -0.014    0   essence   

      year_extracted  car_age  is_new  price_per_fiscal  brand_encoded  \
1659          2022.0   -0.865       0             0.211         -0.670   
100           2024.0   -1.291       0             1.741          1.618   
58            2021.0   -0.652       0             0.235         -0.320   
971           2020.0   -0.439       0             1.375          1.618   
1707          2020.0   -0.439       0            -0.428         -0.709   

      body_type_encoded  model_encoded  
1659            

In [121]:

# nne-hot encode fuel_type
merged_clean = pd.get_dummies(merged_clean, columns=['fuel_type'], prefix='fuel')

# removing redundant columns
columns_to_drop = [
    'year_extracted',    # Replaced by car_age
    'new',               # Replaced by is_new 
]


merged_clean = merged_clean.drop(columns_to_drop, axis=1)

print(f"Final dataset shape: {merged_clean.shape}")
print(f"Final columns: {merged_clean.columns.tolist()}")

print("\nFinal dataset sample:")
print(merged_clean.sample(5).round(3))

Final dataset shape: (5081, 14)
Final columns: ['price', 'kilometrage', 'transmission', 'puissance_fiscale', 'car_age', 'is_new', 'price_per_fiscal', 'brand_encoded', 'body_type_encoded', 'model_encoded', 'fuel_diesel', 'fuel_electrique', 'fuel_essence', 'fuel_hybride']

Final dataset sample:
         price  kilometrage  transmission  puissance_fiscale  car_age  is_new  \
2273   86400.0       -1.352           1.0             -0.015   -1.504       1   
2400  139900.0       -1.352           1.0             -0.015   -1.504       1   
3355   52500.0       -1.348           0.0             -0.015    1.053       0   
6567   85000.0       -0.970           1.0             -0.014   -1.078       0   
736    82000.0        0.392           1.0             -0.014    0.840       0   

      price_per_fiscal  brand_encoded  body_type_encoded  model_encoded  \
2273             0.584         -0.711             -0.140          0.130   
2400             3.568          0.466              1.749          1.2

In [122]:
# Handle the missing values
missing_mask = merged_clean.isnull().any(axis=1)
print(f"rows with missing values: {missing_mask.sum()}")

# Show which rows have missing values
if missing_mask.sum() > 0:
    print("\nRows with missing values:")
    print(merged_clean[missing_mask])
    
    # Remove rows with missing values 
    merged_clean = merged_clean.dropna()
    print(f"\nRemoved {missing_mask.sum()} rows with missing values")
    print(f"Final clean dataset: {merged_clean.shape[0]} rows, {merged_clean.shape[1]} columns")
else:
    print("No missing values found!")

#extract csv
merged_clean.to_csv('car_prices_final_preprocessed.csv', index=False)
print("Final preprocessed dataset sample:")
print(merged_clean.sample(5).round(3))



rows with missing values: 16

Rows with missing values:
         price  kilometrage  transmission  puissance_fiscale   car_age  \
2467  179950.0    -1.351646           NaN          -0.014447 -1.504032   
2675   38000.0     0.430077           NaN          -0.014512  0.413496   
2723   35000.0     0.684609           NaN          -0.014501 -0.225680   
2971   35000.0    -1.349254           NaN          -0.014512  1.478789   
3090   33000.0    -1.348592           NaN          -0.014490  1.478789   
3141   29500.0     0.913688           NaN          -0.014458  1.478789   
3426   35000.0     0.315538           NaN          -0.014490  1.052671   
4279   44000.0    -0.206253           NaN          -0.014522 -0.864856   
4283   75000.0    -0.104440           NaN          -0.014490 -0.012622   
4338   60000.0    -0.218979           NaN          -0.014490 -0.012622   
4359   22500.0     1.193673           NaN          -0.014522  1.904906   
5563  220000.0    -0.460785           NaN          -0.01

# Summary

We merged three datasets (automobile.tn new cars, automobile.tn used cars, and Tayara.tn marketplace listings) and standardized key features, ensuring consistency in brand names and transmission values across all sources. Seats and climatisation columns were excluded as they were not available in all datasets. Irrelevant columns and rare categories were removed, and outliers in price, mileage, and year were filtered out. New features were created, including car age, price per fiscal power, and binary flags for new cars. Categorical variables were encoded using target encoding and one-hot encoding, while numerical features were scaled for modeling. After handling missing values, the final clean dataset combining all three sources is ready for analysis and saved as `car_prices_final_preprocessed.csv`.
